In [14]:
def ead2marc_245(raw):

    '''INDICATORS'''
    '''Indicator 1 is constant (1)'''
    '''Indicator 2 is constant (0)'''

    
    '''SUBFIELD A'''
    '''Get clean title'''
    # (This portion of code was partially revised utilizing ChatGPT 5.2)
    title_fetch = raw.xpath(".//*[local-name()='unittitle']")
    title_raw = title_fetch[0]
    title_clean = title_raw.xpath("string()").strip()
    title_clean = html.escape(title_clean)
    a_245 = f"""<subfield code="a">{title_clean}</subfield>"""
    
    '''PRINT 245 FIELD'''
    field_245_str_nb = """<datafield tag="245" ind1="1" ind2="0">""" + a_245 + "</datafield>"
    field_245_xml = etree.fromstring(field_245_str_nb)
    field_245_str = etree.tostring(field_245_xml, pretty_print=True, encoding="unicode")
    print(field_245_str)
    return field_245_xml

In [ ]:
import re, collections, requests, html
from lxml import etree

marc_rda_relators = {
'''Creates dictionary of relator codes common to MARC and RDA'''
    "ape": "appellee",
    "apl": "appellant",
    "arc": "architect",
    "art": "artist",
    "aup": "audio producer",
    "aus": "screenwriter",
    "aut": "author",
    "bka": "book artist",
    "cad": "casting director",
    "chr": "choreographer",
    "cll": "calligrapher",
    "cmp": "composer",
    "com": "compiler",
    "cou": "court governed",
    "csl": "consultant",
    "ctg": "cartographer",
    "dfd": "defendant",
    "dgc": "degree committee member",
    "dgg": "degree granting institution",
    "dgs": "degree supervisor",
    "drt": "director",
    "dsr": "designer",
    "dte": "dedicatee",
    "dto": "dedicator",
    "edd": "editorial director",
    "enj": "enacting jurisdiction",
    "fmd": "film director",
    "fmk": "filmmaker",
    "fmp": "film producer",
    "his": "host institution",
    "inv": "inventor",
    "isb": "issuing body",
    "ive": "interviewee",
    "ivr": "interviewer",
    "jud": "judge",
    "jug": "jurisdiction governed",
    "lbt": "librettist",
    "lsa": "landscape architect",
    "lyr": "lyricist",
    "med": "medium",
    "orm": "organizer",
    "pht": "photographer",
    "pra": "praeses",
    "prg": "programmer",
    "prn": "production company",
    "pro": "producer",
    "ptf": "plaintiff",
    "rap": "rapporteur",
    "rcp": "addressee",
    "rdd": "radio director",
    "res": "researcher",
    "rpc": "radio producer",
    "rsp": "respondent",
    "rxa": "remix artist",
    "scl": "sculptor",
    "tld": "television director",
    "tlp": "television producer",
    }

'''Gets xml file and sets tree and root'''
tree = etree.parse(r"C:\Users\yello\OneDrive\Documents\EAD2MARC\EAD2MARC_workzone\MC122_ead3.xml")
root = tree.getroot()

'''Checks that document is EAD3'''
if 'http://ead3.archivists.org/schema/' in root.tag:
    
    # '''Strips namespace from xml file'''
    ## (This portion of code was fully generated by ChatGPT 5.2)
    # for elem in root.getiterator():
    #     if not hasattr(elem.tag, 'find'):
    #         continue
    #     i = elem.tag.find('}')
    #     if i >= 0:
    #         elem.tag = elem.tag[i+1:]

    '''Get target <c> tag'''
    # (This portion of code was partially revised utilizing ChatGPT 5.2)
    target_level = input("Enter target hierarchy level: ").strip()

    result = root.xpath(
        "//*[starts-with(local-name(), 'c') and @level=$t_lvl]",
        t_lvl=target_level
    )

    # TODO: change to <c> tags specifically instead of any tag that starts with c??

    '''Set global variables'''
    for i in result:
        n = result.index(i)
        c0_raw = result[n]
        '''Set global names lists'''
        names_list = c0_raw.xpath(".//*[local-name()='origination']") # (Troubleshot using ChatGPT 5.2)

        all_persnames_list = []
        all_corpnames_list = []
        all_famnames_list = []

        creator_names_list = []
        creator_persnames_list = []
        creator_corpnames_list = []
        creator_famnames_list = []

        subject_names_list = []
        subject_persnames_list = []
        subject_corpnames_list = []
        subject_famnames_list = []

        source_names_list = []
        source_persnames_list = []
        source_corpnames_list = []
        source_famnames_list = []

        for orig in names_list:
            if orig.attrib["label"].lower() == "creator":
                creator_names_list.append(orig)
            elif orig.attrib["label"].lower() == "subject":
                subject_names_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_names_list.append(orig)

        for orig in names_list:
            if orig.xpath(".//*[local-name()='persname']"):
                all_persnames_list.append(orig)
                if orig.attrib["label"].lower() == "creator":
                    creator_persnames_list.append(orig)
                elif orig.attrib["label"].lower() == "subject":
                    subject_persnames_list.append(orig)
                elif orig.attrib["label"].lower() == "source":
                    source_persnames_list.append(orig)
            elif orig.xpath(".//*[local-name()='corpname']"):
                all_corpnames_list.append(orig)
                if orig.attrib["label"].lower() == "creator":
                    creator_corpnames_list.append(orig)
                elif orig.attrib["label"].lower() == "subject":
                    subject_corpnames_list.append(orig)
                elif orig.attrib["label"].lower() == "source":
                    source_corpnames_list.append(orig)
            elif orig.xpath(".//*[local-name()='famname']"):
                all_famnames_list.append(orig)
                if orig.attrib["label"].lower() == "creator":
                    creator_famnames_list.append(orig)
                elif orig.attrib["label"].lower() == "subject":
                    subject_famnames_list.append(orig)
                elif orig.attrib["label"].lower() == "source":
                    source_famnames_list.append(orig)
        ead2marc_245(c0_raw)
        # c0_print = etree.tostring(c0_raw, pretty_print=True, encoding="unicode")
        # print (c0_print)

    # print (c0_print)
else:
    '''Returns error message if document is not EAD3'''
    print("Uploaded file must be in EAD3 (not EAD 2002 or any other EAD version). Please upload a new file and try again.")


<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">Crystallization, for symphony orchestra</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">דיאלוגים, לפסנטר ואנסמבל קאמרי = Dialogues, for piano and chamber ensemble</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">Reminiscence, for violin and piano</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">גוונים... לצבעים, לפסנטר ולרביעיית מיתרים = Shades... to colours, for piano and string quartet</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">כסאות במדבר = Chairs in the desert</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">Ciclos, string quartet</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">Jewish-Spanish song cycle, for voice and guitar</subfield>
</datafield>

<datafield tag="245" ind1="1" ind2="0">
  <subf